# Crime classifier trained from scratch
Run notebook 01 first and review labels, annotation schemas, readability, and source IDs. Select Runtime → Change runtime type → T4 GPU, then run cells in order. This notebook defaults to preparing the balanced prototype without training. Full training requires your separate approval. Checkpoints and metrics are saved each epoch to Drive; a disconnected in-progress epoch may be lost. Rerunning training starts fresh from the fixed seed; existing best checkpoints can be evaluated without retraining.
Schema v2: regenerate old manifests. Canonical root deduplication verifies duplicate bytes with SHA-256 and writes excluded_duplicates.csv. Only Burglary, Stealing, Fighting, and Vandalism are mapped. Normal and all other categories remain separately reported as unmapped. No Colab or real-data result is claimed by this notebook.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/AI-Crime-Investigation-FYP')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ranaalihaider99993-source/AI-Crime-Investigation-FYP.git', str(REPO)], check=True)
else:
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Colab checkout has local changes. Preserve them before updating; use a fresh runtime if needed.')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', 'origin/main'], check=True)
os.chdir(REPO)
# Evict previously imported project modules after updating a reused Colab runtime.
for module_name in list(sys.modules):
    if module_name == 'scripts' or module_name.startswith('scripts.'):
        del sys.modules[module_name]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
from scripts.common import seed_all
seed_all(42)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/AI-Crime-Investigation-FYP/canonical-four-class-v2')
except ImportError:
    OUTPUT = REPO / 'artifacts'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Persistent output:', OUTPUT)


In [ ]:
import kagglehub
try:
    DATA = Path(kagglehub.dataset_download('mateohervas/dcsass-dataset'))
except Exception as exc:
    raise RuntimeError('Kaggle download failed. Check network/access; if authentication is required, use Colab secrets or kagglehub.login(), never commit credentials.') from exc
print('Dataset root:', DATA)


In [ ]:
from scripts.data import inspect, split_manifest
from pathlib import Path
import json
CONFIG = json.loads(Path('configs/prototype.json').read_text())
# First real-data smoke only: 3 original sources/class, one clip/source, 2 epochs.
CONFIG.update(sources_per_class=3, epochs=2, batch_size=4)
# Do not enlarge this experiment until the smoke succeeds and full training is approved.
# Rebuild paths for this runtime. Review the new manifest BEFORE enabling training.
report = inspect(DATA, OUTPUT / 'inspection')
print(json.dumps(report, indent=2))
print(CONFIG)


Confirm that all retained clips have the intended class, source IDs identify original videos, and each source has only one class. Flat unknown clip naming needs a human-reviewed source mapping in manifest.csv. Do not mark this review complete based on class names alone. This baseline averages logits from uniformly sampled frames; it does not localize events or model motion. Clip category labels may include normal frames.
With the default three sources/class, the smoke split is one train, one validation, and one test source per class: 12 clips total. Even using all eight readable Fighting sources would give only 6/1/1 per class under this policy. Actual counts are in classifier_splits.report.json; a single held-out Fighting video cannot establish reliable generalization. Neither review nor training flags are automatically enabled.


In [ ]:
SOURCE_MAPPING_REVIEWED = False  # Set True only after reviewing actual manifest/source grouping.
RUN_PROTOTYPE_TRAINING = False  # Enable for a small prototype, not full training.
if SOURCE_MAPPING_REVIEWED:
    splits = split_manifest(OUTPUT / 'inspection/manifest.csv', OUTPUT / 'classifier_splits.csv',
                            seed=CONFIG['seed'], per_class=CONFIG['sources_per_class'])
    print('Selected independent source clips:', len(splits))
    print((OUTPUT / 'classifier_splits.report.json').read_text())
    # Review available counts, actual split_counts and empty source_overlap lists before training.
print('Review classifier_splits.report.json before enabling training. Both flags default to False.')


In [ ]:
if RUN_PROTOTYPE_TRAINING:
    if not SOURCE_MAPPING_REVIEWED:
        raise ValueError('Review canonical source grouping and actual split report before training')
    from scripts.classifier import train
    history = train(OUTPUT / 'classifier_splits.csv', OUTPUT / 'classifier', CONFIG)
else:
    print('Training disabled. Review the split report, then explicitly enable the small smoke run.')


In [ ]:
# Evaluate only after this run trained successfully; never report an old checkpoint as a new run.
if RUN_PROTOTYPE_TRAINING:
    from scripts.classifier import evaluate
    metrics = evaluate(OUTPUT / 'classifier_splits.csv',
                       OUTPUT / 'classifier/crime_classifier.pth',
                       OUTPUT / 'classifier/test_metrics.json')
    print(json.dumps(metrics, indent=2))
    print('Best checkpoint:', OUTPUT / 'classifier/crime_classifier.pth')
    print('This 12-clip (four-class) smoke experiment is not a reliable performance benchmark.')
    # The checkpoint includes state_dict, config and class order for later Django loading.
    DOWNLOAD_CHECKPOINT = False  # Set True to download a copy to your computer.
    if DOWNLOAD_CHECKPOINT:
        from google.colab import files
        files.download(str(OUTPUT / 'classifier/crime_classifier.pth'))
